## Bronze Work for the Novortise Use Case Source Tables

In [0]:
from delta.tables import DeltaTable
from pyspark.sql.window import Window
import uuid
from datetime import datetime
from pyspark.sql.functions import *

In [0]:
%sql
CREATE schema if not EXISTS novortise_catalog.bronze;

#### Creating ingestion_control table for ingestion snapshot of the source tables

In [0]:
%sql
CREATE TABLE IF NOT EXISTS novortise_catalog.bronze.ingestion_control(
  layer string,
  table_name string,
  ts_col STRING,
  pk_col string,
  last_successful_ts TIMESTAMP,
  last_successful_pk BIGINT,
  last_run_id string,
  rows_written BIGINT,
  run_status string,
  updated_at TIMESTAMP
)
using delta

#### Step 3 - Source table configuration

In [0]:
tables_config={
    "orders":{"pk_col":"order_id","ts_col":"updated_at"},
    "products":{"pk_col":"product_id","ts_col":"updated_at"},
    "payments":{"pk_col":"payment_id","ts_col":"processed_at"}
}

bronze_run_id=str(uuid.uuid4())
print("Current Bronze Run ID: ",bronze_run_id)

## Step 4 - Helper functions
This cell contains reusable functions:
- get_last_successful_watermark() reads the latest processes watermark from the control state
- upsert_bronze_control() updates the control table after a successful Bronze load
The fucntions keep the main load logice cleaner and easier to understand

In [0]:
def get_last_successful_watermark(table_name:str):
    ctrl=(
        spark.table("novortise_catalog.bronze.ingestion_control")
        .filter(
            (col("layer")=="bronze") & (col("table_name")==table_name) & (col("run_status")=="success")
        ).orderBy(col("updated_at").desc()).limit(1)
    )
    rows=ctrl.collect()
    if not rows:
        return None,None
    return rows[0]["last_successful_ts"],rows[0]["last_successful_pk"]

In [0]:
def upsert_bronze_control(table_name,ts_col,pk_col,last_ts,last_pk,rows_written,run_id):
    control_df=spark.createDataFrame(
        [(
            "bronze",
            table_name,ts_col,pk_col,last_ts,int(last_pk) if last_pk is not None else None,
            run_id,int(rows_written),"success",datetime.utcnow()
        )],
        schema='''
        layer string,
        table_name string,
        ts_col string,
        pk_col string,
        last_successful_ts timestamp,
        last_successful_pk bigint,
        last_run_id string,
        rows_written bigint,
        run_status string,
        updated_at timestamp
        '''
    )
    dt=DeltaTable.forName(spark,"novortise_catalog.bronze.ingestion_control")
    dt.alias("t").merge(
        control_df.alias("s"),
        expr("t.table_name=s.table_name and t.layer=s.layer")
    ).whenMatchedUpdate(
        set={
            "last_successful_ts":"s.last_successful_ts",
            "last_successful_pk":"s.last_successful_pk",
            "updated_at":"s.updated_at",
            "rows_written":"s.rows_written",
            "run_status":"s.run_status",
            "last_run_id":"s.last_run_id",
            "ts_col":"s.ts_col",
            "pk_col":"s.pk_col"
        }
    ).whenNotMatchedInsertAll().execute()

In [0]:
for table_name,cfg in tables_config.items():
    ts_col=cfg["ts_col"]
    pk_col=cfg["pk_col"]
    last_successful_ts,last_successful_pk=get_last_successful_watermark(table_name)
    source_table=f"`db_novortise_catalog`.dbo.{table_name}"
    target_table=f"novortise_catalog.bronze.{table_name}_raw"
    print(f"\n=== Processing {table_name} ===")
    print(f"Last successful ts:  {last_successful_ts}")
    print(f"Last successful pk: {last_successful_pk}")
    
    source_df=spark.read.table(source_table).withColumn(ts_col,col(ts_col).cast("timestamp"))
    if last_successful_ts is None:
        rows_to_load=source_df 
    else:
        rows_to_load=source_df.filter(
        (col(ts_col)>last_successful_ts) |
        (
            (col(ts_col)==lit(last_successful_ts)) & col(pk_col).cast("long")>lit(int(last_successful_pk))
        )
        )
    rows_to_load=(
        rows_to_load.withColumn("bronze_ingested_at",current_timestamp()).withColumn("bronze_run_id",lit(bronze_run_id)).withColumn("bronze_source_table",lit(source_table))
    )

    row_count=rows_to_load.count()
    print(f"{table_name} rows_to_load = {row_count}")
    
    if row_count==0:
        print(f"No new rows to load for {table_name}")
        upsert_bronze_control(table_name,ts_col,pk_col,last_successful_ts,last_successful_pk,row_count,bronze_run_id)
        continue
    rows_to_load.write.format("delta").mode("append").saveAsTable(target_table)
    max_ts=rows_to_load.agg(max(ts_col).alias("max_ts")).collect()[0]["max_ts"]
    max_pk=rows_to_load.filter(col(ts_col)==lit(max_ts)).agg(max(pk_col).cast("long").alias("max_pk")).collect()[0]["max_pk"]

    upsert_bronze_control(table_name,ts_col,pk_col,max_ts,max_pk,row_count,bronze_run_id)
    print(f"wrote {row_count} rows to {target_table}")

In [0]:
df=spark.read.table("novortise_catalog.bronze.orders_raw")
#df=df.dropDuplicates(["order_id"])
df.count()
